# 1. Indexing -- Sepetix

- from langchain_community.document_loaders import DirectoryLoader, TextLoader
- from langchain_text_splitters import RecursiveCharacterTextSplitter
- from langchain_huggingface import HuggingFaceEmbeddings
- from langchain_chroma import Chroma

- Sepetix şirket belgeleri (kargo, iade, ödeme, uyelik...)

1. docs = DirectoryLoader("sirket-bilgi-bankasi/", glob = "**/*.md", loader_cls = TextLoader).load()

2. splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 60)
3. chunks = splitter.split_documents(docs)

- Retrieval key'siz: local embedding modeli
4. embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
5. store = Chroma.from_documents(chunks, embeddings, persist_directory = "./db")



# Neden RAG? --- RAG'siz vs RAG'li

- RAG'siz

1. llm.invoke("Sepetix Plus yıllık kaç TL?")
- Model bilmez: uydurur
- ya da "Bilgim yok" der

- RAG'li

1. docs = retriever.invoke("Sepetix Plus...")
2. ctx = docs[0].page_content
3. llm.invoke(f"{ctx}\n\nSoru: ....")
- -> "199 TL/yıl" Üyelik & Sadakat

# 2. RAG Chain - Sepetix destek asistanı

- from langchain_openai import ChatOpenAI
- from langchain_core.prompts import ChatPromptTemplate

- retriever = store.as_retriever(k = 4)
- llm = ChatOpenAI(model = "gpt-5.4-mini", temperature = 0)

- PROMPT = ChatPromptTemplate.from_messages([
    ("system",
    "Sen Sepetix'in müşteri destek asistanısın. Sadece aşağıdaki"
    "şirket belgelerinden cevap ver. Belgelerde yoksa"
    "'Bu konuda elimde bilgi yok' de. \n{baglam}"),
    ("user", "{soru}")
])

- def sor(soru, k = 2):
    docs = retriever.invoke(soru)[:k]
    baglam = "\n\n".join(d.page_content for d in docs)
    cevap = llm.invoke(PROMPT.format(baglam = baglam, soru = soru)).content
    return cevap, kaynaklar


# LangChain'in idiomatı

- from langchain_core.runnables import RunnablePassthrough
- from langchain_core.output_parsers import StrOutParser

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"bbaglam": retriever | format_docs, "soru" : RunnablePassthrough()}
)

rag_chain.invoke("Sepetix Plus kaç TL?")

# Özet 

- Load -> Split -> Embed -> Store

- Retriever + Prompt + LLM = RAG

- Sources döndür (güven + doğrulama)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

In [ ]:
docs = DirectoryLoader("sirket-bilgi-bankasi/", glob="**/*.md", loader_cls=TextLoader).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60).split_documents(docs)
KATEGORI = {
    "kargo-teslimat": "Kargo & Teslimat", "iade-degisim": "İade & Değişim",
    "siparis-odeme": "Sipariş & Ödeme", "uyelik-sadakat": "Üyelik & Sadakat",
    "sss": "Sıkça Sorulan Sorular",
}
for ch in chunks:
    ad = os.path.basename(ch.metadata["source"]).replace(".md", "")
    ch.metadata["kategori"] = KATEGORI.get(ad, ad)
load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectordb = Chroma.from_documents(chunks, embedding=embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print("hazır:", len(chunks), "chunk")

In [ ]:
# 1. LLM'i bağla (OpenAI)
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OpenAI key gerekir (.env: OPENAI_API_KEY=sk-...) - embedding ve..."
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# 2. Neden RAG? — RAG'siz vs RAG'li
soru = "Sepetix Plus üyeliği yıllık kaç TL ve avantajları ne?"

# (a) RAG'SİZ - modele doğrudan sor (Sepetix'i bilmez -> uydurur ya da bilemez)
print("🔴 RAG'SİZ:\n", llm.invoke(soru).content, "\n")

# (b) RAG'Lİ - ilgili belgeyi getirip bağlam olarak ver
baglam = "\n\n".join(d.page_content for d in retriever.invoke(soru))
print("🟢 RAG'Lİ:\n", llm.invoke(f"Şu bilgilere göre cevapla:\n{baglam}\n\nSoru: {soru}").content)

In [ ]:
# 3. RAG prompt'u
PROMPT = """Sen Sepetix\'in müşteri destek asistanısın. SADECE aşağıdaki şirket belgelerinden yararla...
Cevap belgelerde yoksa "Bu konuda elimde bilgi yok, müşteri hizmetlerine danışın" de. Kısa ve net ol.

ŞİRKET BELGELERİ:
{baglam}

SORU: {soru}
CEVAP:"""


# 4. Asistan — sor() : cevap + KAYNAK
def sor(soru, k=4):
    docs = retriever.invoke(soru)[:k]
    baglam = "\n\n".join(d.page_content for d in docs)
    cevap = llm.invoke(PROMPT.format(baglam=baglam, soru=soru)).content
    kaynaklar = sorted({d.metadata["kategori"] for d in docs})
    return cevap, kaynaklar

cevap, kaynaklar = sor("Ürünü kaç günde iade edebilirim?")
print(cevap, "\n\n📚 Kaynak:", ", ".join(kaynaklar))

In [ ]:
# Test Döngüsü ve Kapsam Dışı Soru
for s in ["Kapıda ödeme ücretli mi?", "Aynı gün teslimat var mı?"]:
    c, k = sor(s)
    print(f"S: {s}\n   {c}\n   📚 {', '.join(k)}\n")

In [ ]:
# Kapsam dışı soru -> uydurmaz
c, k = sor("iPhone 16 ne zaman satışa çıkacak?")
print(c)

In [ ]:
# 5. LangChain Expression Language (LCEL) - rag_chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def format_docs(docs): return "\n\n".join(d.page_content for d in docs)
prompt_t = ChatPromptTemplate.from_template(PROMPT)

rag_chain = (
    {"baglam": retriever | format_docs, "soru": RunnablePassthrough()}
    | prompt_t | llm | StrOutputParser()
)
print(rag_chain.invoke("Sepetix Plus yıllık kaç TL?"))
```[cite: 1, 2, 3, 4, 5, 6, 7, 8]